# 1.回调功能使用技巧

知识点讲解：Callback 回调机制与链路可观测性

1. 什么是 Callback（回调）
   - LangChain 提供的「事件钩子」机制，在链执行的各个生命周期节点触发回调方法
   - 作用：让链的执行过程从黑盒变成透明，可用于日志、监控、计费、调试
   - 不侵入业务代码：回调通过 config 传入，无需修改链本身的结构

2. BaseCallbackHandler 的核心事件（按组件分类）
   - LLM / ChatModel 类：
       on_llm_start          普通 LLM 开始执行
       on_chat_model_start   聊天模型开始执行（ChatOpenAI 走的是这个）
       on_llm_new_token      流式输出时每产生一个 token 触发
       on_llm_end            模型执行结束，携带完整的 LLMResult
       on_llm_error          模型执行出错
   - Chain 类：on_chain_start / on_chain_end / on_chain_error
   - Tool 类：on_tool_start / on_tool_end / on_tool_error
   - Retriever 类：on_retriever_start / on_retriever_end / on_retriever_error
   - 其他：on_text、on_agent_action、on_agent_finish

3. on_chat_model_start 与 on_llm_start 的区别（常见踩坑点）
   - ChatModel（如 ChatOpenAI）触发的是 on_chat_model_start，不是 on_llm_start
   - 若只实现 on_llm_start，使用 ChatOpenAI 时回调不会被触发
   - 但结束事件是共用的：ChatModel 结束时触发的仍是 on_llm_end

4. 回调方法的通用参数说明
   - serialized: Dict - 组件的序列化信息（类名、模块路径、参数等）
   - run_id: UUID - 本次运行的唯一标识，可用于串联同一次调用的各个事件
   - parent_run_id: Optional[UUID] - 父级运行 ID，据此可还原出完整的调用树
   - tags / metadata - 通过 config 传入的自定义标签与元数据，便于分类统计
   - **kwargs - 预留扩展参数，子类实现必须保留，否则新版本新增参数时会报错

5. 回调的三种注册方式（作用域不同）
   - 请求级：chain.invoke(input, config={"callbacks": [handler]})，仅本次调用生效
   - 组件级：ChatOpenAI(model=..., callbacks=[handler])，该组件的所有调用都生效
   - 全局级：langchain_core.globals.set_debug(True) 或环境变量 LANGCHAIN_TRACING_V2

6. 内置回调处理器
   - StdOutCallbackHandler：把链的进入/退出信息打印到标准输出，最简单的调试工具
   - ConsoleCallbackHandler：输出更详细的彩色树形结构日志
   - FileCallbackHandler：把回调日志写入文件
   - LangSmith Tracer：上报到 LangSmith 平台，提供完整可视化追踪

7. 本示例实现的 LLMOps 场景
   - on_chat_model_start 记录起始时间戳 self.start_at，并打印输入消息
   - on_llm_end 计算 end_at - start_at 得到本次调用耗时
   - 这是 LLMOps 平台统计「响应延迟、token 消耗、成本」的基础能力

8. 注意事项
   - Handler 实例中保存状态（如 start_at）在并发场景下会互相覆盖，
     生产环境应以 run_id 为 key 存入字典，或每次请求新建 Handler 实例
   - 回调方法中的异常默认会被吞掉（raise_error=False），调试时需注意
   - 流式调用必须消费完生成器，on_llm_end 才会被触发（本例末尾的 for 循环即为此目的）


In [ ]:
import time
from typing import Dict, Any, List, Optional
from uuid import UUID
import dotenv
from langchain_core.callbacks import StdOutCallbackHandler, BaseCallbackHandler
from langchain_core.messages import BaseMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.outputs import LLMResult
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI
dotenv.load_dotenv()

In [ ]:
class LLMOpsCallbackHandler(BaseCallbackHandler):
    """
    自定义LLMOps回调处理器
    设计目的：
        统计单次大模型调用的耗时，并记录输入消息与完整输出，
        这是 LLMOps 平台做性能监控、成本核算的最小可用实现。
    继承关系：
        BaseCallbackHandler 定义了全部生命周期钩子的默认空实现，
        子类按需覆写关心的事件方法即可，无需实现全部方法。
    """
    start_at: float = 0
    def on_chat_model_start(
            self,
            serialized: Dict[str, Any],
            messages: List[List[BaseMessage]],
            *,
            run_id: UUID,
            parent_run_id: Optional[UUID] = None,
            tags: Optional[List[str]] = None,
            metadata: Optional[Dict[str, Any]] = None,
            **kwargs: Any,
    ) -> Any:
        """
        on_chat_model_start(...) -> Any
          作用：聊天模型（ChatModel）开始执行时触发的钩子
          参数：
            serialized: Dict[str, Any] - 模型的序列化信息，包含 id（类路径）、
                                         kwargs（模型参数如 model 名称、temperature）
            messages: List[List[BaseMessage]] - 发送给模型的消息批次。
                                         外层 list 对应 batch 中的每条输入，
                                         内层 list 是该条输入的消息序列
            run_id: UUID - 本次模型运行的唯一标识
            parent_run_id: Optional[UUID] - 父级运行 ID（这里是外层 chain 的 run_id）
            tags: Optional[List[str]] - config 中传入的标签列表
            metadata: Optional[Dict[str, Any]] - config 中传入的元数据
            **kwargs: Any - 预留扩展参数，必须保留以兼容未来版本
          返回：Any - 返回值不被框架使用，通常返回 None
          副作用：记录开始时间戳到 self.start_at，供 on_llm_end 计算耗时
          重要提示：ChatOpenAI 触发的是本方法，而不是 on_llm_start，
                   只实现 on_llm_start 会导致回调静默失效
        """
        print("聊天模型开始执行了")
        print("serialized:", serialized)
        print("messages:", messages)
        self.start_at = time.time()
    def on_llm_end(
            self,
            response: LLMResult,
            *,
            run_id: UUID,
            parent_run_id: Optional[UUID] = None,
            **kwargs: Any,
    ) -> Any:
        """
        on_llm_end(response: LLMResult, ...) -> Any
          作用：LLM 与 ChatModel 执行结束时共用的钩子（结束事件不区分两种模型）
          参数：
            response: LLMResult - 模型的完整输出结果对象，关键字段：
                        - generations: List[List[Generation]] 生成结果的二维列表
                        - llm_output: dict 模型返回的附加信息，
                                      通常包含 token_usage（prompt_tokens /
                                      completion_tokens / total_tokens）与 model_name
                        - run: 运行信息列表
            run_id: UUID - 本次运行的唯一标识，与 on_chat_model_start 的 run_id 一致
            parent_run_id: Optional[UUID] - 父级运行 ID
            **kwargs: Any - 预留扩展参数
          返回：Any - 返回值不被框架使用
          副作用：打印完整响应与本次调用耗时
          触发时机：流式调用时，必须把生成器消费完毕本事件才会触发
        """
        end_at: float = time.time()
        print("完整输出:", response)
        print("程序消耗:", end_at - self.start_at)

In [ ]:
prompt = ChatPromptTemplate.from_template("{query}")
llm = ChatOpenAI(model="deepseek-flash")
chain = {"query": RunnablePassthrough()} | prompt | llm | StrOutputParser()
resp = chain.stream(
    "你好，你是？",
    config={"callbacks": [StdOutCallbackHandler(), LLMOpsCallbackHandler()]}
)
for chunk in resp:
    pass